In [1]:
import pytesseract
import numpy as np
import cv2
import pandas as pd
from collections import defaultdict
import pymupdf

In [18]:
doc = pymupdf.open("../docs/input_data.pdf")  # open document

images = []

for i, page in enumerate(doc):
    pix = page.get_pixmap(dpi=300) 
    images.append(np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n))
img = images[0]

In [19]:
img.shape

(3300, 2550, 3)

In [ ]:
def preprocess_ocr_image(img):
    image_cv = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    image_cv = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
    image_cv = cv2.GaussianBlur(image_cv, (3, 3), 0)
    blur_for_sharpen = cv2.GaussianBlur(image_cv, (9, 9), 10.0)
    image_cv = cv2.addWeighted(image_cv, 1.5, blur_for_sharpen, -1, 0)
    _, image_cv = cv2.threshold(image_cv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    image_cv = cv2.morphologyEx(image_cv, cv2.MORPH_CLOSE, kernel)

    return image_cv


In [14]:
preprocessed_img = preprocess_ocr_image(img)

In [15]:
data = pytesseract.image_to_data(preprocessed_img, output_type=pytesseract.Output.DICT)

In [ ]:

lines = defaultdict(list)

for i in range(len(data['text'])):
    if data['text'][i].strip():
        key = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
        lines[key].append((data['left'][i], data['text'][i].strip()))

sorted_lines = []
for line_id in sorted(lines):
    words = sorted(lines[line_id], key=lambda x: x[0])
    line_text = " ".join([w[1] for w in words])
    sorted_lines.append(line_text)

# Output lines
for line in sorted_lines:
    print(line)

ey RE « sutrorzation/ Prior Authorization Request Form : ~
vp te “Complete all Sections to ensure timely review ---...: ao :
.. aU t service(s) are related to Cancer, Specialty Medications or Behavioral Health please use ‘the designated form 4
*All preauthorization requests must be submitted with supporting clinical documentation that is relevant to the request.
Forms will be returned if not fille ingly or if they are submitted withaut the required clinical information.
Fax to r for Urgent Services fax t
Provider appeals submitted on this form will not be considered, Please use the claim resubmission request form found on or our ir website. -
Section A: Request information ee he ee ST DE a DT is
Today's Date: 98/29/2022 Completed _ |
Decisions on preauthorization requests submitted with all necessary clinical information will be made within 15 calendar days of receipt of the request.
Bi Service is Scheduled (only if applicable) Schedule Date: 10/21/2022
DO Urgent Request (only if appli

In [20]:
import cv2

for i in range(len(data['text'])):
    if int(data['conf'][i]) > 30:
        (x, y, w, h) = (data['left'][i], data['top'][i], data['width'][i], data['height'][i])
        text = data['text'][i].strip()
        if text:
            cv2.rectangle(preprocessed_img, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(preprocessed_img, text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (36,255,12), 1)

cv2.imwrite("ocr_debug_boxes.png", preprocessed_img)

True

In [6]:
%%writefile ../modules/ocr.py

import pytesseract
import numpy as np
import cv2
import pandas as pd
from collections import defaultdict
import pymupdf

class OCRPipeline:
    def __init__(self, doc_path):
        self.doc_path = doc_path
        self.text_data = {}

    def preprocess_ocr_image(self, img):
        image_cv = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        image_cv = cv2.cvtColor(image_cv, cv2.COLOR_BGR2GRAY)
        image_cv = cv2.GaussianBlur(image_cv, (3, 3), 0)
        blur_for_sharpen = cv2.GaussianBlur(image_cv, (9, 9), 10.0)
        image_cv = cv2.addWeighted(image_cv, 1.5, blur_for_sharpen, -1, 0)
        _, image_cv = cv2.threshold(image_cv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        image_cv = cv2.morphologyEx(image_cv, cv2.MORPH_CLOSE, kernel)
        return image_cv

    def ocr_page(self, page):
        pix = page.get_pixmap(dpi=300)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        preprocessed_img = self.preprocess_ocr_image(img)

        data = pytesseract.image_to_data(preprocessed_img, output_type=pytesseract.Output.DICT)
        lines = defaultdict(list)

        for i in range(len(data['text'])):
            if data['text'][i].strip():
                key = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
                lines[key].append((data['left'][i], data['text'][i].strip()))

        sorted_lines = []
        for line_id in sorted(lines):
            words = sorted(lines[line_id], key=lambda x: x[0])
            line_text = " ".join([w[1] for w in words])
            sorted_lines.append(line_text)

        return " ".join(sorted_lines)

    def run(self):
        try:
            doc = pymupdf.open(self.doc_path)

            for i, page in enumerate(doc):
                page_number = i + 1
                text = page.get_text().strip()

                if text:
                    self.text_data[page_number] = text
                else:
                    print(f"No text on page {page_number}, applying OCR.")
                    self.text_data[page_number] = self.ocr_page(page)

            return self.text_data

        except Exception as e:
            print(f"[Error] {e}")
            return None



Overwriting ../modules/ocr.py


In [3]:
pipeline = OCRPipeline("../docs/input_data.pdf")
data = pipeline.run()

[Info] No text on page 1, applying OCR.
[Info] No text on page 2, applying OCR.
[Info] No text on page 3, applying OCR.
[Info] No text on page 4, applying OCR.
[Info] No text on page 5, applying OCR.


In [4]:
data

{1: 'as. Authorization/ Prior Authorization Request Form, : : has ts oo, : Complete all Sections to ensure timely review . a vi f service(s) are related to Cancer, Specialty Medications or Behavioral Health please u use e the ciesignated form Sons *All preauthorization requests must be submitted with supporting clinical documentation that is relevant to the request. Forms will be returned if not fille izaly or if they are submitted withaut the required clinical information. Fax to r for Urgent Services fax t Provider appeals submitted on this form will nat be considered. Please use e the claim resubmission request form found or on our ir website, Section A: Request Information “vs OE PU oe Today\'s Date: 08/29/2022 Completed _ Decisions on preauthorization requests submitted with all necessary clinical intormation will be made within 15 calendar days of receipt of the request. BB Service is Scheduled (only if applicable) Schedule Date: 10/21/2022 0 urgent Request (ently if applicable) 

In [34]:
import pickle
with open("data.pkl", "wb") as f:
    pickle.dump(data, f)